In [4]:
import pandas as pd
import ast

portfolio = pd.read_csv('../../data/portfolio.csv')
profile = pd.read_csv('../../data/profile.csv')
transcript = pd.read_csv('../../data/transcript.csv')

In [5]:
# 복사
transcript_df = transcript.copy()

# value → dict
transcript_df['value_dict'] = transcript_df['value'].apply(ast.literal_eval)

# 컬럼 추출
transcript_df['offer_id'] = transcript_df['value_dict'].apply(
    lambda x: x.get('offer id', x.get('offer_id'))
)
transcript_df['amount'] = transcript_df['value_dict'].apply(lambda x: x.get('amount'))
transcript_df['reward'] = transcript_df['value_dict'].apply(lambda x: x.get('reward'))

# customer_id 통일
if 'person' in transcript_df.columns:
    transcript_df = transcript_df.rename(columns={'person': 'customer_id'})

In [6]:
# 1) offer completed
completed_df = transcript_df[transcript_df['event'] == 'offer completed'][
    ['customer_id', 'offer_id', 'time', 'reward']
].rename(columns={'time': 'completed_time'})

# 2) transaction
transaction_df = transcript_df[transcript_df['event'] == 'transaction'][
    ['customer_id', 'time', 'amount']
].rename(columns={'time': 'transaction_time'})

# 3) offer received (참고용)
received_df = transcript_df[transcript_df['event'] == 'offer received'][
    ['customer_id', 'offer_id', 'time']
].rename(columns={'time': 'received_time'})

In [ ]:
# 고객 기준 merge
check_df = completed_df.merge(
    transaction_df,
    on='customer_id',
    how='left'
)

# completed 이전 거래(동시에도 포함) -> True에 해당하면 difficulty(프로모션 조건 충족을 위한 최소 지출 금액) 충족을 위한 거래 완료 상태
check_df['before_completed'] = (
    check_df['transaction_time'] <= check_df['completed_time']
)

# completed 이후 거래
check_df['after_completed'] = (
    check_df['transaction_time'] > check_df['completed_time']
)

In [16]:
check_df[check_df['before_completed'] == True]

,customer_id,offer_id,completed_time,reward,transaction_time,amount,before_completed,after_completed
0,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,2906b810c7d4411798c6938adc9daaa5,0,2.0,0,34.56,True,False
12,fe97aa22dd3e48c8b143116a8403dd52,fafdcd668e3743c1bb461111dcafc2a4,0,2.0,0,18.97,True,False
23,629fc02d56414d91bca360decdfa9288,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,5.0,0,33.90,True,False
26,676506bad68e4161b9bbaffeb039626b,ae264e3637204a6fb9bb56bc8210ddfd,0,10.0,0,18.01,True,False
34,8f7dd3b2afe14c078eb4f6e6fe4ba97d,4d5c57ea9a6940dd891ad53e9dbe8da0,0,10.0,0,19.11,True,False
...,...,...,...,...,...,...,...,...
333777,8431c16f8e1d440880db371a68f82dd0,fafdcd668e3743c1bb461111dcafc2a4,714,2.0,714,1.19,True,False
333778,24f56b5e1849462093931b164eb803b5,fafdcd668e3743c1bb461111dcafc2a4,714,2.0,138,21.39,True,False
333779,24f56b5e1849462093931b164eb803b5,fafdcd668e3743c1bb461111dcafc2a4,714,2.0,150,14.12,True,False
333780,24f56b5e1849462093931b164eb803b5,fafdcd668e3743c1bb461111dcafc2a4,714,2.0,264,16.92,True,False


In [17]:
summary = check_df.groupby(['customer_id', 'offer_id', 'completed_time'], as_index=False).agg(
    has_tx_before=('before_completed', 'max'),
    has_tx_after=('after_completed', 'max'),
    tx_count=('transaction_time', 'count')
)

summary.head()

,customer_id,offer_id,completed_time,has_tx_before,has_tx_after,tx_count
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576,True,True,8
1,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414,True,True,8
2,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528,True,True,8
3,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,576,True,True,5
4,0011e0d4e6b944f998e987f904e8c1e5,2298d6c36e964ae4a3e7e9706d1fb8c2,252,True,True,5


In [18]:
summary[['has_tx_before', 'has_tx_after']].value_counts()

has_tx_before  has_tx_after
True           True            29409
               False            3773
Name: count, dtype: int64

In [19]:
# completed인데 이전 거래 있는 경우 비율
no_before = summary['has_tx_before'].value_counts(normalize=True)
print(no_before)

has_tx_before
True    1.0
Name: proportion, dtype: float64


offer completed는 “조건을 만족하는 구매가 이미 발생한 상태”를 의미한다
따라서 completed는 별도의 행동이 아니라 transaction 이후 발생하는 이벤트
조건 충족 구매는 transaction_time <= completed_time으로 봐야 한다

completed 이후 transaction은 추가 구매(업셀/재방문)를 의미한다
따라서 퍼널과 전환 정의는 아래처럼 분리해야 한다:

[프로모션 효과]
received → viewed → completed

[구매 발생]
received → transaction

[추가 소비]
completed → transaction